# Module 5 — AI Agents: Topics & Practice

## Learning outcomes
1. Implement a ReAct loop from scratch — no framework.
2. Compose a multi-step graph with LangGraph including conditional edges + checkpointing.
3. Design multi-agent hand-offs that dont loop forever.
4. Add the guardrails that turn a demo into a production agent.

## 1. The agent loop

Every agent, regardless of framework:
```
while not done:
    thought  = llm(state)            # plan / reflect
    action   = pick_tool(thought)    # decide
    obs      = run_tool(action)      # act
    state    = update(state, obs)    # remember
```
The framework choices (LangChain agents, LangGraph, CrewAI, OpenAI Assistants) differ in how they encode `state`, `update`, and stopping rules.

> **Exercise 5.1** — Implement the loop in <60 lines of pure Python with two tools (calculator, web_search). Cap iterations at 10. Add a `done` tool the model calls to halt.

## 2. ReAct in detail

ReAct interleaves *Reason* and *Act* with this template:
```
Thought: I need to find ...
Action: search
Action Input: "..."
Observation: <tool output>
Thought: Now I should ...
...
Final Answer: ...
```
The structured trace gives you (a) interpretability, (b) easy stopping criteria, (c) a place to inject critics.

> **Exercise 5.2** — Take Exercise 5.1s loop. Add a `Critic` LLM that reviews each `Thought` and returns `{accept | revise}`. Show how this reduces wrong actions.

## 3. LangGraph — when a loop becomes a graph

When you need: branching, parallel work, checkpoints, human-in-the-loop, *use a graph*.

Concepts:
- **State** — typed dict, every node mutates it.
- **Nodes** — pure functions of state.
- **Edges** — conditional routing on state.
- **Checkpointer** — persist state per `thread_id`; enables resume + HITL.

> **Exercise 5.3** — Convert Ex 5.2s ReAct loop into a LangGraph with nodes `[planner, executor, critic]` and a conditional edge that returns to `planner` if the critic rejects.

## 4. Multi-agent — patterns

| Pattern | When |
|---|---|
| **Supervisor → workers** | Clear hierarchy, supervisor routes |
| **Hand-off** | Specialists pass control with shared context |
| **Debate** | Two agents argue; a judge decides |
| **Pipeline** | Linear stages (Planner → Researcher → Writer) |

Pitfalls: infinite hand-offs, context bloat, conflicting instructions. Always: *step budget, token budget, message-count cap*.

> **Exercise 5.4** — Pick one pattern. Diagram the message flow for *"plan a 3-day trip to Lisbon"*. Where does it loop? How do you cap it?

## 5. Guardrails

Production agents need:
- **Budget caps** — `$ + tokens + steps` per run.
- **Tool allow-lists** — per-user, per-context.
- **Output filters** — PII, secrets, profanity.
- **Prompt-injection defence** — never let tool outputs influence system prompt.
- **Observability** — every node, every tool call, end-to-end trace.

> **Exercise 5.5** — Add to your Ex 5.3 graph: (a) a $0.50 cost cap that aborts the run, (b) an OTLP/Langfuse trace exporter.